# 🦙 Local RAG with Ollama

The same RAG pipeline as `simple-rag-demo.ipynb` — but this time the generation step is **real**.  
No API keys. No cloud. Everything runs on your machine.

**Stack:**
- `sentence-transformers` — embed documents (all-MiniLM-L6-v2)
- `faiss-cpu` — vector similarity search
- `ollama` — local LLM for the generation step

**Requires:** [Ollama](https://ollama.com) installed and running locally (`ollama serve`).

> **Colab users:** Ollama can run in Colab — the setup cell below handles it automatically.


## Setup — Install Ollama (Colab only)

If you're running **locally** and Ollama is already installed, **skip this cell**.


In [ ]:
# Only needed in Google Colab — skip if running locally with Ollama already installed

import subprocess, time

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if is_colab():
    print("Colab detected — installing dependencies...")
    subprocess.run(["apt-get", "install", "-y", "-q", "zstd"], check=True)
    print("Colab detected — installing Ollama...")
    subprocess.run(["bash", "-c", "curl -fsSL https://ollama.com/install.sh | sh"], check=True)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)
    print("Ollama server started")
else:
    print("Local environment — assuming 'ollama serve' is running.")
    print("If not, open a terminal and run: ollama serve")

## Install Python dependencies


In [ ]:
!pip install -q sentence-transformers faiss-cpu ollama
print("Dependencies ready")

## Choose your model

| Model | Size | Good for |
|-------|------|----------|
| `tinyllama` | ~600 MB | Fastest, lowest RAM — good for testing |
| `phi3.5` | ~2.2 GB | Compact Microsoft model, surprisingly smart |
| `llama3.2:1b` | ~1.3 GB | Meta's smallest Llama 3 |
| `llama3.2:3b` | ~2 GB | Better quality, still fast |
| `mistral` | ~4 GB | Strong reasoning, popular choice |

Edit `MODEL` below. First run pulls the model (takes a few minutes, then cached).


In [ ]:
import ollama

MODEL = "tinyllama"  # change me: phi3.5 | llama3.2:1b | llama3.2:3b | mistral

print(f"Pulling {MODEL}... (first time may take a few minutes)")
ollama.pull(MODEL)
print(f"{MODEL} is ready")

## Step 1 — Build the Knowledge Base

Same 8-document set as `simple-rag-demo.ipynb`. Extend it with your own content.


In [ ]:
documents = [
    "RAG stands for Retrieval-Augmented Generation. It combines search with LLMs.",
    "FAISS is a library by Meta for efficient similarity search on dense vectors.",
    "Sentence transformers convert text into meaningful numerical embeddings.",
    "Local RAG runs entirely on your machine — no API calls, no data leaving your system.",
    "Ollama lets you run models like Llama 3 or Mistral locally with one command.",
    "Vector databases store embeddings and support fast approximate nearest-neighbor search.",
    "The retrieval step finds the top-k most relevant chunks from your knowledge base.",
    "Chunking splits long documents into smaller pieces before embedding them.",
]

print(f"📚 {len(documents)} documents loaded")

## Step 2 — Embed Documents

`all-MiniLM-L6-v2` is ~80 MB and runs fine on CPU.


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(documents, convert_to_numpy=True)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Each document → {embeddings.shape[1]}-dimensional vector")

## Step 3 — Index with FAISS


In [ ]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"FAISS index built — {index.ntotal} vectors indexed")

## Step 4 — Retrieve

Find the top-k most relevant documents for any query.


In [ ]:
def retrieve(query: str, top_k: int = 3) -> list[str]:
    query_vec = embedder.encode([query], convert_to_numpy=True)
    _, indices = index.search(query_vec, top_k)
    return [documents[i] for i in indices[0]]

# Quick test
test_query = "How does RAG work?"
results = retrieve(test_query)
print(f"Query: {test_query}\n")
print("Top 3 retrieved docs:")
for i, doc in enumerate(results, 1):
    print(f"  {i}. {doc}")

## Step 5 — Generate with Ollama 🦙

This is the step that was missing in `simple-rag-demo.ipynb`.

We pass the retrieved context to the local LLM and get a **real answer** — grounded in the knowledge base, not hallucinated from training data.


In [ ]:
def rag(query: str, top_k: int = 3) -> str:
    context_docs = retrieve(query, top_k)
    context = "\n".join(f"- {doc}" for doc in context_docs)

    prompt = f"""You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say exactly: "I don't have enough context to answer this."

Context:
{context}

Question: {query}
Answer:"""

    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"]


q = "How does RAG work?"
print(f"Q: {q}")
print(f"A: {rag(q)}")

## Step 6 — Try It Yourself


In [ ]:
my_query = "What is FAISS and who made it?"  # change me

print(f"Q: {my_query}")
print(f"A: {rag(my_query)}")

## Step 7 — The Hallucination Test

Ask something that is **not** in the knowledge base. Does the model admit it doesn't know — or make something up?

This is the core challenge from the [human RAG exercise](human-rag-exercise.ipynb): grounding.


In [ ]:
unanswerable = "What is the capital of France?"

print(f"Q: {unanswerable}  (not in our knowledge base)")
print(f"A: {rag(unanswerable)}")
print()
print("Did the model stay grounded, or did it answer from training data anyway?")
print("Change the system prompt to make the model more or less strict.")

## Bonus — Streaming Output

Print tokens as they arrive instead of waiting for the full response.


In [ ]:
def rag_stream(query: str, top_k: int = 3):
    context_docs = retrieve(query, top_k)
    context = "\n".join(f"- {doc}" for doc in context_docs)

    prompt = f"""Answer using ONLY the context below.

Context:
{context}

Question: {query}
Answer:"""

    print(f"Q: {query}\nA: ", end="", flush=True)
    for chunk in ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}], stream=True):
        print(chunk["message"]["content"], end="", flush=True)
    print()


rag_stream("What does FAISS do?")

## 🏋️ Exercises

1. **Add your own documents** — replace the `documents` list with content from your domain
2. **Switch models** — change `MODEL = "mistral"` and compare answer quality
3. **Tighten the grounding prompt** — make the model refuse to answer anything not in context
4. **100% local stack** — replace SentenceTransformers with `ollama.embeddings(model="nomic-embed-text", prompt=text)`  
   Pull it first: `!ollama pull nomic-embed-text`
5. **Load real documents** — use `pathlib` to read `.txt` files from a folder and embed them

---

**Comparison:**

| | `simple-rag-demo.ipynb` | `simple-rag-demo-ollama.ipynb` |
|--|--|--|
| Embedding | SentenceTransformers | SentenceTransformers |
| Vector search | FAISS | FAISS |
| Generation | ❌ prompt string only | ✅ Ollama local LLM |
| Internet required | ✅ (model download once) | ❌ fully offline after setup |
| Data privacy | ✅ local | ✅ local |
